Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

# Nepali Sign Language Detection
Here we are using the mediapipe package to create an ML


The MediaPipe Model Maker package is used for creating our gesture model. Our dataset consists of all the vowels and consonants in the devanagri language.

## Importing

In [ ]:
!pip install --upgrade pip
!pip install mediapipe-model-maker
!pip install mediapipe
!pip install tensorflow
!pip install matplotlib
!pip install scikit-learn


In [ ]:
from google.colab import drive
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt
import numpy as np
import mediapipe as mp
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Import dataset


We create a folder each of the categories in addition to one for none, which contains all of the invalid cases.

dataset format: `<dataset_path>/<label_name>/<img_name>.*`.

while using collab online we can only upload whole files at a time, so we first zip the whole file.


In [ ]:
!unzip /content/drive/MyDrive/NSL_folder/model/model_boosted.zip
dataset_path = "model_boosted"

In [ ]:
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(labels)

**Training the session**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

Split the data into 80% for training, 10% for validation, and 10% for testing

# Data Preprocessing and Augmentation
# Set up ImageDataGenerator for data augmentation

In [ ]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

**Train the model**

adding the hyperparameters into the code and starting training


In [ ]:
hparams = gesture_recognizer.HParams(export_dir="exported_model")
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

**Evaluating performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

need to add


*   confusion matrix
*   Recall
*   F1 score
*   Precision
*   Cross Entropy
*   Prediction Score
*   



In [ ]:

loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

In [ ]:
model.export_model()
!ls exported_model

In [ ]:
print(f"Number of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(validation_data)}")
print(f"Number of test samples: {len(rest_data)}")


In [ ]:
!unzip /test_data_files.zip
test_data_path="test_data_files"

In [ ]:
# Apply the model on new data
filename = "test_data_files/म_45.jpg"  # Replace with your test image path
base_options = mp.tasks.BaseOptions(
    model_asset_path=hparams.export_dir + "/gesture_recognizer.task"
)
options = mp.tasks.vision.GestureRecognizerOptions(
    base_options=base_options, running_mode=mp.tasks.vision.RunningMode.IMAGE
)

with mp.tasks.vision.GestureRecognizer.create_from_options(options) as recognizer:
    mp_image = mp.Image.create_from_file(str(filename))
    result = recognizer.recognize(mp_image)

# Process and display results
print(f"Predicted gesture: {result.gestures[0][0].category_name}")


In [ ]:
import os
import tqdm
import pandas as pd
import numpy as np
import mediapipe as mp

# Path to the test data folder containing images
test_data_path = "test_data_files"  # Update with your actual path if different

# List all image files in the test_data_path directory
testfiles = [os.path.join(test_data_path, f) for f in os.listdir(test_data_path) if os.path.isfile(os.path.join(test_data_path, f))]

# Initialize result list
test_results = []

# Create the GestureRecognizer and process each test file
with mp.tasks.vision.GestureRecognizer.create_from_options(options) as recognizer:
    for filename in tqdm.tqdm(testfiles):
        mp_image = mp.Image.create_from_file(str(filename))
        result = recognizer.recognize(mp_image)

        if len(result.gestures) > 0:
            pred = result.gestures[0][0].category_name or "n/a"
        else:
            pred = "empty"

        # Extract label from the filename
        label = os.path.basename(filename).split("_")[0]  # Example: "थ_1.jpg"

        test_results.append((filename, label, pred))

# Create DataFrame from results
results_df = pd.DataFrame(test_results, columns=["filename", "label", "pred"])

# Display the DataFrame
print(results_df.head())

# Checking unique labels and predictions
unique_labels = results_df['label'].unique()
unique_predictions = results_df['pred'].unique()
print("Unique Labels:", unique_labels)
print("Unique Predictions:", unique_predictions)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import sklearn.metrics


# Checking unique labels and predictions
unique_labels = results_df['label'].unique()
unique_predictions = results_df['pred'].unique()
print("Unique Labels:", unique_labels)
print("Unique Predictions:", unique_predictions)

# Exclude unwanted labels
exclude_labels = ["none", "n/a", "empty"]
results_df_filtered = results_df[~results_df['label'].isin(exclude_labels)]
results_df_filtered = results_df_filtered[~results_df_filtered['pred'].isin(exclude_labels)]

# Extract unique labels from filtered results
unique_labels_filtered = results_df_filtered['label'].unique().tolist()
unique_predictions_filtered = results_df_filtered['pred'].unique().tolist()

# Combine with unique labels from results_df to ensure all labels are included
classes = sorted(list(set(unique_labels_filtered + unique_predictions_filtered)))

cm = sklearn.metrics.confusion_matrix(results_df_filtered["label"], results_df_filtered["pred"], labels=classes, normalize="true")

# Specify the path to Noto Sans Devanagari font
font_path = './NotoSansDevanagari.ttf'  # Update with the actual path
prop = fm.FontProperties(fname=font_path)

# Plot confusion matrix with specified font
plt.rcParams['font.family'] = prop.get_name()  # Set the custom font family
plt.rcParams['font.size'] = 10  # Adjust font size as needed

# Display the confusion matrix
disp = sklearn.metrics.ConfusionMatrixDisplay(cm * 100, display_labels=classes)
disp.plot(include_values=False)
plt.show()


In [ ]:
print(results_df.head())

In [ ]:
!pip install seaborn

import seaborn as sns

results_df["result"] = np.where(
    results_df.pred == results_df.label,
    "correct",
    np.where(results_df.pred.isin(["n/a", "empty"]), "not found", "incorrect"),
)
print(results_df.result.value_counts(normalize=True))
sns.histplot(
    data=results_df, x="label", hue="result", multiple="stack", stat="count"
)

In [ ]:
import numpy as np
import sklearn.metrics as metrics

# Example labels and predictions (replace with your actual data)
labels = results_df["label"]
predictions = results_df["pred"]

# Compute precision, recall, and F1 score
precision, recall, f1_score, _ = metrics.precision_recall_fscore_support(labels, predictions, average='weighted')

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1_score:.2f}")


In [ ]:
import pandas as pd
import re

# Define empty lists to store extracted data
epochs = []
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
learning_rates = []

# Provided block of text
console_output = """
None
Epoch 1/10
102/102 [==============================] - 5s 28ms/step - loss: 0.2082 - categorical_accuracy: 0.7304 - val_loss: 0.0187 - val_categorical_accuracy: 1.0000 - lr: 0.0010
Epoch 2/10
102/102 [==============================] - 2s 20ms/step - loss: 0.1895 - categorical_accuracy: 0.7255 - val_loss: 0.0293 - val_categorical_accuracy: 0.9615 - lr: 9.9000e-04
Epoch 3/10
102/102 [==============================] - 2s 18ms/step - loss: 0.1753 - categorical_accuracy: 0.7353 - val_loss: 0.0301 - val_categorical_accuracy: 0.9615 - lr: 9.8010e-04
Epoch 4/10
102/102 [==============================] - 2s 15ms/step - loss: 0.1758 - categorical_accuracy: 0.7549 - val_loss: 0.0219 - val_categorical_accuracy: 0.9615 - lr: 9.7030e-04
Epoch 5/10
102/102 [==============================] - 2s 19ms/step - loss: 0.1568 - categorical_accuracy: 0.7500 - val_loss: 0.0315 - val_categorical_accuracy: 0.9615 - lr: 9.6060e-04
Epoch 6/10
102/102 [==============================] - 2s 20ms/step - loss: 0.1481 - categorical_accuracy: 0.7647 - val_loss: 0.0178 - val_categorical_accuracy: 1.0000 - lr: 9.5099e-04
Epoch 7/10
102/102 [==============================] - 2s 18ms/step - loss: 0.1530 - categorical_accuracy: 0.7402 - val_loss: 0.0286 - val_categorical_accuracy: 0.9615 - lr: 9.4148e-04
Epoch 8/10
102/102 [==============================] - 2s 20ms/step - loss: 0.1436 - categorical_accuracy: 0.7696 - val_loss: 0.0232 - val_categorical_accuracy: 0.9615 - lr: 9.3207e-04
Epoch 9/10
102/102 [==============================] - 3s 27ms/step - loss: 0.1398 - categorical_accuracy: 0.7696 - val_loss: 0.0212 - val_categorical_accuracy: 0.9615 - lr: 9.2274e-04
Epoch 10/10
102/102 [==============================] - 2s 20ms/step - loss: 0.1360 - categorical_accuracy: 0.7647 - val_loss: 0.0233 - val_categorical_accuracy: 0.9615 - lr: 9.1352e-04"""

# Split the console output into lines
lines = console_output.strip().splitlines()

# Define regular expressions to extract relevant information
epoch_pattern = re.compile(r'Epoch (\d+)/\d+')
metrics_pattern = re.compile(r'loss: (\d+\.\d+) - categorical_accuracy: (\d+\.\d+) - val_loss: (\d+\.\d+) - val_categorical_accuracy: (\d+\.\d+) - lr: (\d+\.\d+e-\d+)')

# Process each line
for line in lines:
    # Match epoch number
    epoch_match = epoch_pattern.match(line)
    if epoch_match:
        epoch = int(epoch_match.group(1))

    # Match metrics (losses, accuracies, learning rate)
    metrics_match = metrics_pattern.search(line)
    if metrics_match:
        train_loss = float(metrics_match.group(1))
        train_acc = float(metrics_match.group(2))
        val_loss = float(metrics_match.group(3))
        val_acc = float(metrics_match.group(4))
        lr = float(metrics_match.group(5))

        # Append values to lists
        epochs.append(epoch)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)
        learning_rates.append(lr)

# Create a DataFrame
training_data = pd.DataFrame({
    "Epoch": epochs,
    "Train Loss": train_losses,
    "Val Loss": val_losses,
    "Train Accuracy": train_accuracies,
    "Val Accuracy": val_accuracies,
    "Learning Rate": learning_rates
})

# Display the DataFrame
print(training_data)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure that the columns are properly formatted as numpy arrays
epochs = np.array(training_data['Epoch'])
train_losses = np.array(training_data['Train Loss'])
val_losses = np.array(training_data['Val Loss'])

# Plotting training and validation losses
plt.figure(figsize=(10, 6))

# Training loss plot
plt.plot(epochs, train_losses, marker='o', label='Train Loss')

# Validation loss plot
plt.plot(epochs, val_losses, marker='o', label='Val Loss')

# Adding labels and title
plt.title('Training and Validation Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Set y-axis ticks with increments of 0.05
y_min = min(train_losses.min(), val_losses.min()) - 0.05
y_max = max(train_losses.max(), val_losses.max()) + 0.05
plt.yticks(np.arange(y_min, y_max, 0.05))

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:
# Ensure that the accuracy columns are also properly formatted as numpy arrays
train_accuracies = np.array(training_data['Train Accuracy'])
val_accuracies = np.array(training_data['Val Accuracy'])

# Plotting training and validation accuracies
plt.figure(figsize=(10, 6))

# Training accuracy plot
plt.plot(epochs, train_accuracies, marker='o', label='Train Accuracy')

# Validation accuracy plot
plt.plot(epochs, val_accuracies, marker='o', label='Val Accuracy')

# Adding labels and title
plt.title('Training and Validation Accuracies')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Set y-axis ticks with increments of 0.05 for accuracies
y_min_acc = min(train_accuracies.min(), val_accuracies.min()) - 0.05
y_max_acc = max(train_accuracies.max(), val_accuracies.max()) + 0.05
plt.yticks(np.arange(y_min_acc, y_max_acc, 0.05))

# Display the plot
plt.tight_layout()
plt.show()


**Export to Tensorflow Lite Model**

Now exporting it into the .task file used by mediapipe.

Note:The export also includes model metadata, which includes the label file.

In [ ]:
model.export_model()
!ls exported_model

In [ ]:
files.download('exported_model/gesture_recognizer.task')

## Note on Hyperparameters {:#hyperparameters}


You can further customize the model using the `GestureRecognizerOptions` class, which has two optional parameters for `ModelOptions` and `HParams`. Use the `ModelOptions` class to customize parameters related to the model itself, and the `HParams` class to customize other parameters related to training and saving the model.

`ModelOptions` has one customizable parameter that affects accuracy:
* `dropout_rate`: The fraction of the input units to drop. Used in dropout layer. Defaults to 0.05.
* `layer_widths`: A list of hidden layer widths for the gesture model. Each element in the list will create a new hidden layer with the specified width. The hidden layers are separated with BatchNorm, Dropout, and ReLU. Defaults to an empty list(no hidden layers).

`HParams` has the following list of customizable parameters which affect model accuracy:
* `learning_rate`: The learning rate to use for gradient descent training. Defaults to 0.001.
* `batch_size`: Batch size for training. Defaults to 2.
* `epochs`: Number of training iterations over the dataset. Defaults to 10.
* `steps_per_epoch`: An optional integer that indicates the number of training steps per epoch. If not set, the training pipeline calculates the default steps per epoch as the training dataset size divided by batch size.
* `shuffle`: True if the dataset is shuffled before training. Defaults to False.
* `lr_decay`: Learning rate decay to use for gradient descent training. Defaults to 0.99.
* `gamma`: Gamma parameter for focal loss. Defaults to 2

Additional `HParams` parameter that does not affect model accuracy:
* `export_dir`: The location of the model checkpoint files and exported model files.

For example, the following trains a new model with the dropout_rate of 0.2 and learning rate of 0.003.

In [ ]:
hparams = gesture_recognizer.HParams(learning_rate=0.005, epochs=20, export_dir="exported_model_3")
model_options = gesture_recognizer.ModelOptions(dropout_rate=0.05)
options = gesture_recognizer.GestureRecognizerOptions(model_options=model_options, hparams=hparams)
model_3 = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Evaluate the newly trained model.

In [ ]:
loss, accuracy = model_3.evaluate(test_data)
print(f"Test loss:{loss}, Test accuracy:{accuracy}")